In [ ]:
!pip install torch transformers pandas numpy scikit-learn sentencepiece

In [ ]:
!pip install wandb --quiet

In [ ]:
import wandb
wandb.login(key="7f36a41a8c5d6b661c74c15a7229f9807271df9b")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from transformers import (AutoTokenizer, AutoModel, AutoConfig, Trainer,
                          TrainingArguments, EarlyStoppingCallback, default_data_collator)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import numpy as np
# data_path = "/content/drive/My Drive/NLP/Midterm/nlp-2025-midterm-kaggle-asas/"
data_path = "/kaggle/input/nlp-2025-midterm-kaggle-asas/"

In [ ]:
train_df = pd.read_csv(data_path + "train.csv")
test_df = pd.read_csv(data_path + "test.csv")
submission_df = pd.read_csv(data_path + "sample_submission.csv")

In [ ]:
train_df['text'] = train_df['question'] + " " + train_df['answer']
test_df['text'] = test_df['question'] + " " + test_df['answer']

In [ ]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['score'].tolist(),
    test_size=0.2,
    random_state=42
)

In [ ]:
class ScoringDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256, augmentation=False):
        self.texts = texts
        self.labels = labels  # สำหรับ test ให้เป็น None
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.augmentation = augmentation  # ไม่ใช้ augmentation ในที่นี้
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        inputs = self.tokenizer(text, truncation=True, padding='max_length',
                                max_length=self.max_length, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        # Extra feature: ความยาวของข้อความ (จำนวนคำ)
        item["text_length"] = torch.tensor(len(text.split()), dtype=torch.float)
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

In [ ]:
def custom_collate_fn(features):
    batch = default_data_collator(features)
    if "text_length" in batch:
        if len(batch["text_length"].shape) == 1:
            batch["text_length"] = batch["text_length"].unsqueeze(1)
    return batch

In [ ]:
class CustomRegressionModel(nn.Module):
    def __init__(self, model_name, dropout_rate=0.5, extra_features_dim=16):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name, num_labels=1, problem_type="regression")
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout_rate)
        # Extra feature layer: แปลง scalar "text_length" เป็น vector ด้วย dimension ที่กำหนด
        self.extra_feature_layer = nn.Sequential(
            nn.Linear(1, extra_features_dim),
            nn.ReLU()
        )
        # Regression head รับ input จาก CLS token และ extra feature
        self.regressor = nn.Linear(self.config.hidden_size + extra_features_dim, 1)
    
    def forward(self, input_ids, attention_mask, token_type_ids=None, text_length=None, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]  # shape: (batch, hidden_size)
        
        if text_length is not None:
            # ตรวจสอบมิติของ text_length: ถ้าเป็น 1D ให้ unsqueeze เป็น (batch, 1)
            if text_length.dim() == 1:
                text_length = text_length.unsqueeze(1)
            extra_features = self.extra_feature_layer(text_length)  # ควรได้ shape: (batch, extra_features_dim)
            combined = torch.cat([cls_output, extra_features], dim=1)
        else:
            combined = cls_output
        
        combined = self.dropout(combined)
        logits = self.regressor(combined).squeeze(-1)
        
        if labels is None:
            loss = torch.tensor(0.0, device=input_ids.device)
        else:
            loss_fct = nn.MSELoss()
            loss = loss_fct(logits, labels.float())
        return {"loss": loss, "logits": logits}


In [ ]:
model_name = "clicknext/phayathaibert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = CustomRegressionModel(model_name, dropout_rate=0.5, extra_features_dim=16)

# สร้าง Dataset (ไม่ใช้ augmentation)
train_dataset = ScoringDataset(train_texts, train_labels, tokenizer, max_length=256, augmentation=False)
val_dataset = ScoringDataset(val_texts, val_labels, tokenizer, max_length=256, augmentation=False)
test_dataset = ScoringDataset(test_df['text'].tolist(), None, tokenizer, max_length=256, augmentation=False)

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.squeeze()
    mse = mean_squared_error(labels, predictions)
    return {"mse": mse}

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=1e-6,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=80,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="mse",
    greater_is_better=False,
    logging_steps=50,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=custom_collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)]
)

In [ ]:
trainer.train()

In [ ]:
eval_results = trainer.evaluate()
print("Evaluation results:", eval_results)

In [ ]:
# ดึง test dataloader จาก Trainer
test_dataloader = trainer.get_test_dataloader(test_dataset)
all_preds = []

# ประมวลผลแต่ละ batch โดยไม่ใช้ trainer.predict โดยตรง
model.eval()
with torch.no_grad():
    for batch in test_dataloader:
        # ย้ายข้อมูลไปยัง device ที่ใช้งาน (เช่น GPU)
        batch = {k: v.to(model.transformer.device) for k, v in batch.items()}
        outputs = model(**batch)
        # outputs["logits"] ควรมี shape (batch_size,)
        batch_preds = outputs["logits"].detach().cpu().numpy()
        all_preds.append(batch_preds)

# รวม predictions จากทุก batch
all_preds = np.concatenate(all_preds, axis=0)

# ตรวจสอบจำนวน predictions
print("Total predictions before slicing:", all_preds.shape)

# ถ้ามี predictions เกินกว่าจำนวนตัวอย่างใน test dataset ให้ตัด slice
if len(all_preds) > len(test_dataset):
    all_preds = all_preds[:len(test_dataset)]
    
print("Final predictions shape:", all_preds.shape)
print("Test dataset length:", len(test_dataset))



In [ ]:
# นำผลลัพธ์ไปสร้าง submission
submission_df['score'] = all_preds
submission_df.to_csv("submission.csv", index=False)
print("Submission file saved as submission.csv")

In [ ]:
# !rm -rf ./results